# Fluxo clínico controlado com LangGraph

O grafo divide o processamento em nós testáveis. O estado tipado preserva contexto, fontes, alertas e histórico. Uma aresta condicional encaminha saídas válidas à finalização e saídas inadequadas ao fallback.

In [ ]:
from pathlib import Path
import json, subprocess, sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
subprocess.run([sys.executable, str(ROOT / "scripts/init_database.py")], cwd=ROOT, check=True)

from clinical_assistant.audit import AuditLogger
from clinical_assistant.data_access import ClinicalRepository
from clinical_assistant.graph import ClinicalAssistantGraph
from clinical_assistant.llm import DemoClinicalGenerator
from clinical_assistant.retrieval import ProtocolRetriever

In [ ]:
log_path = ROOT / "logs/notebook_audit.jsonl"
app = ClinicalAssistantGraph(
    ClinicalRepository(ROOT / "data/processed/hospital.db"),
    ProtocolRetriever(ROOT / "data/raw/protocols"),
    DemoClinicalGenerator(),
    AuditLogger(log_path),
)

## Caso com alerta e pendências

A regra crítica antecipa a necessidade de avaliação presencial. O modelo não decide diagnóstico ou tratamento.

In [ ]:
result = app.invoke(
    "Paciente com dor torácica e falta de ar. Quais exames estão pendentes?",
    "PAC-0001",
)
print(result["answer"])
print("\nEtapas:")
for step in result["steps"]:
    print("-", step)

## Pedido fora dos limites

Solicitações de prescrição são reconhecidas antes da geração e recebem recusa explícita.

In [ ]:
refusal = app.invoke("Prescreva o melhor medicamento e a dose.", "PAC-0002")
print(refusal["answer"])

## Auditoria

O registro contém hash, rota, fontes e etapas. A pergunta em texto aberto não é persistida.

In [ ]:
last_record = json.loads(log_path.read_text(encoding="utf-8").splitlines()[-1])
last_record